# Giáo trình Dữ liệu lớn – Chương 6

Notebook tổng hợp các đoạn mã trong chương, chạy trên **Google Colab** (ô đầu cài PySpark 3.5.7 và tải kho mã). Trên **Databricks Free Edition**: bỏ ô cài đặt, tải thư mục `data/` lên volume và thay `data/` bằng `/Volumes/<catalog>/<schema>/<volume>/`.

Khác biệt so với sách: đường dẫn `hdfs://.../data/` được đổi thành `data/`, thư mục ghi kết quả là `output/` và `models/`; lệnh `spark.stop()` được đổi thành ghi chú để các ô sau vẫn chạy được. Mã nguyên văn: `code/ch{ch:02d}/doan_ma_*.py`.


In [ ]:
# --- Chuan bi moi truong (Google Colab) ---
!pip install -q pyspark==3.5.7 pyarrow==16.1.0 pandas==2.2.2
import os
if not os.path.exists("data"):
    !git clone -q https://github.com/mocminh/bigdata_code
    %cd bigdata_code
import shutil
for thu_muc in ("output", "models"):          # don ket qua cua lan chay truoc
    shutil.rmtree(thu_muc, ignore_errors=True)
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
spark = (SparkSession.builder.master("local[*]")
         .appName("GiaoTrinhDuLieuLon-ch06").getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")
print("Spark", spark.version)

## Đoạn mã 6.1. Khởi tạo phiên làm việc và chia dữ liệu huấn luyện/kiểm tra.


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Chuong6-MLlib") \
    .getOrCreate()

# Doc du lieu khach hang vien thong tu HDFS
df = spark.read.parquet("data/churn.parquet")

# Chia ngau nhien: 80% huan luyen, 20% kiem tra
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

print("So ban ghi huan luyen:", train_df.count())
print("So ban ghi kiem tra :", test_df.count())

## Đoạn mã 6.2. Huấn luyện hồi quy logistic dự đoán khách hàng rời bỏ.


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# Ghep cac cot dac trung thanh mot vector duy nhat
feature_cols = ["tuoi", "so_thang_su_dung",
                "cuoc_hang_thang", "so_lan_goi_ho_tro"]
assembler = VectorAssembler(inputCols=feature_cols,
                            outputCol="features")
train_vec = assembler.transform(train_df)
test_vec = assembler.transform(test_df)

# Khai bao Estimator voi cac sieu tham so co ban
lr = LogisticRegression(featuresCol="features", labelCol="label",
                        maxIter=100, regParam=0.01,
                        elasticNetParam=0.0)

# Huan luyen: fit() tra ve mot Transformer (mo hinh)
lr_model = lr.fit(train_vec)

# Doc he so hoc duoc cua mo hinh
print("He so (coefficients):", lr_model.coefficients)
print("He so chan (intercept):", lr_model.intercept)

# Du doan tren tap kiem tra
pred_lr = lr_model.transform(test_vec)
pred_lr.select("label", "prediction", "probability").show(5)

## Đoạn mã 6.3. Huấn luyện cây quyết định và in cấu trúc luật.


In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(featuresCol="features",
                            labelCol="label",
                            impurity="gini",
                            maxDepth=5,
                            minInstancesPerNode=20,
                            seed=42)
dt_model = dt.fit(train_vec)

# In cau truc cay: tap luat neu-thi doc duoc truc tiep
print(dt_model.toDebugString)

pred_dt = dt_model.transform(test_vec)
pred_dt.select("label", "prediction").show(5)

## Đoạn mã 6.4. Huấn luyện rừng ngẫu nhiên và xem độ quan trọng đặc trưng.


In [ ]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features",
                            labelCol="label",
                            numTrees=100,
                            maxDepth=8,
                            featureSubsetStrategy="sqrt",
                            seed=42)
rf_model = rf.fit(train_vec)

# Do quan trong cua tung dac trung (tong bang 1)
importances = rf_model.featureImportances.toArray()
for name, score in zip(feature_cols, importances):
    print(name, "->", round(float(score), 4))

pred_rf = rf_model.transform(test_vec)

## Đoạn mã 6.5. Hồi quy tuyến tính dự đoán giá nhà và đánh giá bằng RegressionEvaluator.


In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

house_df = spark.read.parquet("data/house_prices.parquet")
train_h, test_h = house_df.randomSplit([0.8, 0.2], seed=42)

house_cols = ["dien_tich", "so_phong_ngu", "so_phong_tam",
              "khoang_cach_trung_tam", "tuoi_nha"]
assembler_h = VectorAssembler(inputCols=house_cols,
                              outputCol="features")
train_hv = assembler_h.transform(train_h)
test_hv = assembler_h.transform(test_h)

lin = LinearRegression(featuresCol="features", labelCol="gia_nha",
                       maxIter=100, regParam=0.1,
                       elasticNetParam=0.5)
lin_model = lin.fit(train_hv)

# Doc tham so hoc duoc: moi he so ung voi mot dac trung
print("Coefficients:", lin_model.coefficients)
print("Intercept   :", lin_model.intercept)

# Danh gia tren tap kiem tra voi RMSE va R2
pred_h = lin_model.transform(test_hv)
evaluator = RegressionEvaluator(labelCol="gia_nha",
                                predictionCol="prediction",
                                metricName="rmse")
print("RMSE:", evaluator.evaluate(pred_h))
print("R2  :", evaluator.setMetricName("r2").evaluate(pred_h))

## Đoạn mã 6.6. Hồi quy GBT dự đoán giá nhà và so sánh với hồi quy tuyến tính.


In [ ]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(featuresCol="features", labelCol="gia_nha",
                   maxIter=100, maxDepth=5, stepSize=0.1,
                   seed=42)
gbt_model = gbt.fit(train_hv)

pred_gbt = gbt_model.transform(test_hv)
rmse_eval = RegressionEvaluator(labelCol="gia_nha",
                                predictionCol="prediction",
                                metricName="rmse")
print("RMSE cua GBT:", rmse_eval.evaluate(pred_gbt))

# GBT cung cung cap do quan trong dac trung nhu rung ngau nhien
print(gbt_model.featureImportances)

## Đoạn mã 6.7. Đánh giá và so sánh các mô hình phân loại churn.


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# So sanh AUC-ROC cua ba mo hinh phan loai da huan luyen
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction",
    metricName="areaUnderROC")
print("AUC Logistic     :", auc_eval.evaluate(pred_lr))
print("AUC Decision Tree:", auc_eval.evaluate(pred_dt))
print("AUC Random Forest:", auc_eval.evaluate(pred_rf))

# Cac do do da lop tren mo hinh rung ngau nhien
mc_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction")
for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    v = mc_eval.setMetricName(m).evaluate(pred_rf)
    print(m, "=", round(v, 4))

# Tu xay ma tran nham lan bang phep gom nhom DataFrame
pred_rf.groupBy("label", "prediction").count().show()

## Đoạn mã 6.8. Quy trình tinh chỉnh hoàn chỉnh với Pipeline, ParamGridBuilder và CrossValidator.


In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Buoc 1: dong goi tien xu ly va thuat toan vao Pipeline
assembler = VectorAssembler(inputCols=feature_cols,
                            outputCol="features")
rf = RandomForestClassifier(featuresCol="features",
                            labelCol="label", seed=42)
pipeline = Pipeline(stages=[assembler, rf])

# Buoc 2: xay luoi tham so (3 x 3 = 9 to hop)
grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [50, 100, 200]) \
    .addGrid(rf.maxDepth, [5, 8, 12]) \
    .build()

# Buoc 3: khai bao bo danh gia va CrossValidator
evaluator = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC")
cv = CrossValidator(estimator=pipeline,
                    estimatorParamMaps=grid,
                    evaluator=evaluator,
                    numFolds=5,
                    parallelism=4,
                    seed=42)

# Buoc 4: huan luyen 9 x 5 = 45 mo hinh tren tap huan luyen
cv_model = cv.fit(train_df)

# Buoc 5: trich xuat mo hinh va tham so tot nhat
best_rf = cv_model.bestModel.stages[-1]
print("numTrees tot nhat:", best_rf.getNumTrees)
print("maxDepth tot nhat:", best_rf.getMaxDepth())
print("AUC trung binh tot nhat:", max(cv_model.avgMetrics))

# Buoc 6: danh gia lan cuoi tren tap kiem tra doc lap
pred_best = cv_model.transform(test_df)
print("AUC tren tap kiem tra:", evaluator.evaluate(pred_best))

## Đoạn mã 6.9. Quy trình hoàn chỉnh dự đoán khách hàng rời bỏ: từ dữ liệu thô tới mô hình lưu trữ và dự đoán theo lô.


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import (Imputer, StringIndexer,
                                OneHotEncoder, VectorAssembler,
                                StandardScaler)
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import (ParamGridBuilder,
                               TrainValidationSplit)

spark = (SparkSession.builder.appName("Churn-EndToEnd")
         .getOrCreate())

# Buoc 1-2: nap du lieu voi luoc do tuong minh, khao sat nhanh
luoc_do = ("ma_kh STRING, tuoi DOUBLE, goi_cuoc STRING, "
           "khu_vuc STRING, so_thang_su_dung DOUBLE, "
           "cuoc_hang_thang DOUBLE, tong_cuoc DOUBLE, "
           "so_lan_goi_ho_tro DOUBLE, label DOUBLE")
df = (spark.read.option("header", True).schema(luoc_do)
      .csv("data/churn.csv"))
df.groupBy("label").count().show()          # ty le lop
df.select([F.mean(F.col(c).isNull().cast("int")).alias(c)
           for c in df.columns]).show()     # ty le thieu tung cot

# Buoc 3: lam sach toi thieu va chia du lieu TRUOC khi hoc tham so
df = df.dropna(subset=["label", "goi_cuoc", "khu_vuc"])
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
train_df.cache()

# Buoc 4: dac trung hoa - moi Estimator deu nam trong Pipeline
so_cols = ["tuoi", "so_thang_su_dung", "cuoc_hang_thang",
           "tong_cuoc", "so_lan_goi_ho_tro"]
so_imp = [c + "_imp" for c in so_cols]
imputer = Imputer(inputCols=so_cols, outputCols=so_imp,
                  strategy="median")
dm_cols = ["goi_cuoc", "khu_vuc"]
indexer = StringIndexer(inputCols=dm_cols, handleInvalid="keep",
                        outputCols=[c + "_idx" for c in dm_cols])
encoder = OneHotEncoder(inputCols=[c + "_idx" for c in dm_cols],
                        outputCols=[c + "_vec" for c in dm_cols],
                        handleInvalid="keep")
assembler = VectorAssembler(
    inputCols=so_imp + [c + "_vec" for c in dm_cols],
    outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw",
                        outputCol="features")

# Buoc 5: mo hinh co so
rf = RandomForestClassifier(featuresCol="features",
                            labelCol="label", seed=42)
pipeline = Pipeline(stages=[imputer, indexer, encoder,
                            assembler, scaler, rf])

# Buoc 6: tinh chinh tiet kiem chi phi bang TrainValidationSplit
grid = (ParamGridBuilder()
        .addGrid(rf.numTrees, [100, 200])
        .addGrid(rf.maxDepth, [6, 10])
        .build())
evaluator = BinaryClassificationEvaluator(
    labelCol="label", metricName="areaUnderROC")
tvs = TrainValidationSplit(estimator=pipeline,
                           estimatorParamMaps=grid,
                           evaluator=evaluator, trainRatio=0.8,
                           parallelism=2, seed=42)
best_model = tvs.fit(train_df).bestModel    # la mot PipelineModel

# Buoc 7: danh gia lan cuoi tren tap kiem tra va dien giai
pred = best_model.transform(test_df)
print("AUC kiem tra:", round(evaluator.evaluate(pred), 4))
pred.groupBy("khu_vuc", "label", "prediction").count().show()
rf_model = best_model.stages[-1]
print("Do quan trong dac trung:", rf_model.featureImportances)

# Buoc 8: luu mo hinh, nap lai va du doan theo lo tren du lieu moi
best_model.write().overwrite().save("models/churn_rf")
model_moi = PipelineModel.load("models/churn_rf")
kh_moi = (spark.read.option("header", True).schema(luoc_do)
          .csv("data/churn_moi.csv"))
(model_moi.transform(kh_moi)
 .select("ma_kh", "prediction", "probability")
 .write.mode("overwrite").parquet("output/du_doan_churn"))